# Model every channel's next dollar

*Data Analysis in Python · Topic 16.2*

Your CMO's budget for next quarter is fixed, so the question is where the next dollar brings the most revenue. Fit the model with the control for demand for every channel group, and collect each one's average return, marginal return and interval in one table.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell, the cell that builds the clean `sessions` table (lesson 3.'s steps) and the cell that builds the `channel_days` table (topic 16.1.'s steps).
2. In the first cell marked ✏️, start with Paid Search:
   - `search_weekly`: one row per full Monday-to-Sunday week of the `channel_days` table's Paid Search rows, with the dates as the index and the columns `spend`, `revenue` and `organic`
   - `search_model`: a regression of `search_weekly`'s revenue on spend and organic

   Its `spend` coefficient should be topic 10.1.'s answer, about 1.983.
3. In the second cell marked ✏️, do the same for every channel group in a loop, and create the `returns` table: one row per channel group, with `channel_group` as the index and the columns:
   - `roas`: the weekly table's revenue divided by its spend
   - `marginal_return`: the model's `spend` coefficient
   - `return_low` and `return_high`: the coefficient's 95% confidence interval

   Sort the table by `marginal_return`, highest first.
4. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: The `search_weekly` table has 104 rows, and `search_model`'s `spend` coefficient is about `1.983`, as in topic 10.1. The `returns` table has the four channel groups, from Paid Search's marginal return of about 1.98 down to Paid Social's of about 0.55.

## Parameters

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = [f"{year}-{month:02d}" for year in (2024, 2025) for month in range(1, 13)]
START, END = "2024-01-01", "2025-12-31"
# The media plan's budget lines, by the third part of a campaign's name: Facebook and Instagram share one
CHANNEL_GROUPS = {"search": "Paid Search", "display": "Display", "facebook": "Paid Social", "instagram": "Paid Social", "youtube": "Paid Video"}

In [ ]:
# The clean sessions table, with lesson 3's steps. Run it, and leave it as it is.
DAY_FIRST_MONTHS = ["2025-03"]
SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}


def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["source"] = sessions["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sessions.shape

In [ ]:
# Spend, revenue and organic sessions per channel group and day, with topic 16.1's steps. Run it, and leave it as it is.
def campaign_key(names):
    """Return campaign names stripped of spaces and lowercased, so they match across systems."""
    return names.str.strip().str.lower()


google = pd.read_csv(DATA_URL + "google_ads_spend.csv", parse_dates=["date"])
meta = pd.read_csv(DATA_URL + "meta_ads_spend.csv", parse_dates=["Reporting starts"])
youtube = pd.read_csv(DATA_URL + "youtube_spend.csv", parse_dates=["Day"])
google["spend"] = google["cost_micros"] / 1_000_000
spend = pd.concat([
    google.rename(columns={"campaign_name": "campaign"})[["date", "campaign", "spend"]],
    meta.rename(columns={"Reporting starts": "date", "Campaign name": "campaign", "Amount spent (USD)": "spend"})[["date", "campaign", "spend"]],
    youtube.rename(columns={"Day": "date", "Campaign": "campaign", "Cost": "spend"})[["date", "campaign", "spend"]],
], ignore_index=True)
spend["campaign"] = campaign_key(spend["campaign"])
spend = spend.groupby(["date", "campaign"], as_index=False)["spend"].sum()

revenue = sessions.groupby(["date", "session_campaign"], as_index=False)["revenue"].sum()
revenue = revenue.rename(columns={"session_campaign": "campaign"})
revenue["campaign"] = campaign_key(revenue["campaign"])
campaign_days = spend.merge(revenue, on=["date", "campaign"], how="left", validate="one_to_one")
campaign_days["channel_group"] = campaign_days["campaign"].str.split("_").str[2].map(CHANNEL_GROUPS)

organic = sessions[sessions["session_medium"] == "organic"].groupby("date", as_index=False)["sessions"].sum()
organic = organic.rename(columns={"sessions": "organic"})
channel_days = campaign_days.groupby(["channel_group", "date"], as_index=False)[["spend", "revenue"]].sum()
channel_days = channel_days.merge(organic, on="date", how="left", validate="many_to_one")

checks = {
    "every campaign is in a channel group": bool(campaign_days["channel_group"].notna().all()),
    "every campaign day has revenue": bool(campaign_days["revenue"].notna().all()),
    "every channel group has every day": len(channel_days) == len(set(CHANNEL_GROUPS.values())) * len(pd.date_range(START, END)),
}
assert all(checks.values()), checks
channel_days.shape

## One channel group first

In [ ]:
# ✏️ Create search_weekly and search_model


## Every channel group

In [ ]:
# ✏️ Create the returns table


In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    def number(value):
        return pd.api.types.is_number(value) and not isinstance(value, (pd.DataFrame, pd.Series))

    search_weekly, search_model = found.get("search_weekly"), found.get("search_model")
    params = getattr(search_model, "params", None)
    coefficient = params.get("spend") if isinstance(params, pd.Series) else None
    if not isinstance(search_weekly, pd.DataFrame) or not {"spend", "revenue", "organic"} <= set(search_weekly.columns):
        results.append((False, "search_weekly should be a DataFrame with the columns spend, revenue and organic, one row per full week of Paid Search."))
    elif len(search_weekly) == 731:
        results.append((False, "search_weekly has 731 rows: one per day. Total the days by week with resample(\"W\"), after set_index(\"date\")."))
    elif len(search_weekly) == 105:
        results.append((False, "search_weekly has 105 rows: the last one is December 29 to 31, 2025, three days. Keep the weeks with 7 days, as in topic 10.1."))
    elif len(search_weekly) != 104:
        results.append((False, f"search_weekly should have the 104 full weeks of 2024 and 2025. It has {len(search_weekly):,} rows."))
    elif abs(search_weekly["spend"].sum() - 305812.10) > 0.05:
        total = search_weekly["spend"].sum()
        hint = " That's every channel group's spend: keep the rows whose channel_group is Paid Search." if total > 500000 else ""
        results.append((False, f"search_weekly's spend adds up to ${total:,.2f}. Paid Search's full weeks add up to $305,812.10.{hint}"))
    elif not number(coefficient):
        results.append((False, "search_model should be a fitted model: smf.ols(\"revenue ~ spend + organic\", data=search_weekly).fit()."))
    elif abs(coefficient - 3.192564) < 0.0001:
        results.append((False, "search_model's spend coefficient is 3.193: that's spend alone. Add the control for demand: \"revenue ~ spend + organic\"."))
    elif abs(coefficient - 1.983305) > 0.0001:
        results.append((False, f"search_model's spend coefficient should be topic 10.1.'s answer, about 1.983. It is {coefficient:.4f}."))
    else:
        results.append((True, "Paid Search on its own: each extra dollar comes with $1.98, topic 10.1.'s answer. The data and the model are right, so the loop can use them."))

    returns = found.get("returns")
    columns = {"roas", "marginal_return", "return_low", "return_high"}
    # Each channel group's ROAS (from full weeks, or from every day) and the range of marginal returns that a model
    # with a control for demand gives: full weeks or days, organic or all sessions as the control
    expected = {
        "Paid Search": ((2.381396, 2.386050), 1.5, 2.5),
        "Paid Video": ((1.618471, 1.619177), 0.8, 1.6),
        "Display": ((1.604192, 1.609629), 0.4, 1.3),
        "Paid Social": ((2.021792, 2.019364), -0.3, 1.0),
    }
    if not isinstance(returns, pd.DataFrame) or not columns <= set(returns.columns):
        results.append((False, "returns should be a DataFrame with the columns roas, marginal_return, return_low and return_high, one row per channel group."))
    elif set(returns.index) != set(expected):
        results.append((False, f"returns should have the four channel groups as its index: {', '.join(expected)}. Its index is {list(returns.index)}: use set_index(\"channel_group\")."))
    elif list(returns.index) != list(returns.sort_values("marginal_return", ascending=False).index):
        results.append((False, "Sort the returns table by marginal_return, highest first: sort_values(\"marginal_return\", ascending=False)."))
    elif any(not any(abs(returns.loc[group, "roas"] - value) < 0.0001 for value in roas) for group, (roas, low, high) in expected.items()):
        wrong = next(group for group, (roas, low, high) in expected.items() if not any(abs(returns.loc[group, "roas"] - value) < 0.0001 for value in roas))
        results.append((False, f"{wrong}'s roas is {returns.loc[wrong, 'roas']:.4f}. It should be the weekly table's revenue divided by its spend, {expected[wrong][0][0]:.4f}."))
    elif returns.loc["Paid Search", "marginal_return"] > returns.loc["Paid Search", "roas"]:
        results.append((False, f"Paid Search's marginal return is {returns.loc['Paid Search', 'marginal_return']:.2f}, more than its ROAS of {returns.loc['Paid Search', 'roas']:.2f}: the model has no control for demand, so the spend gets the credit for the season (topic 10.1.). Use \"revenue ~ spend + organic\"."))
    elif any(not (returns.loc[group, "return_low"] < returns.loc[group, "marginal_return"] < returns.loc[group, "return_high"]) for group in expected):
        results.append((False, "Each row's return_low and return_high should be the 95% interval around its marginal_return: low, high = model.conf_int().loc[\"spend\"]."))
    elif any(not (low <= returns.loc[group, "marginal_return"] <= high) for group, (roas, low, high) in expected.items()):
        wrong = next(group for group, (roas, low, high) in expected.items() if not (low <= returns.loc[group, "marginal_return"] <= high))
        results.append((False, f"{wrong}'s marginal return is {returns.loc[wrong, 'marginal_return']:.3f}. The spend coefficient of revenue ~ spend + organic on its full weeks is about {dict(zip(expected, [1.983, 1.225, 0.873, 0.555]))[wrong]}."))
    else:
        top, bottom = returns.index[0], returns.index[-1]
        results.append((True, f"The next dollar brings the most in {top}, ${returns.loc[top, 'marginal_return']:.2f}, and the least in {bottom}, ${returns.loc[bottom, 'marginal_return']:.2f}, though {bottom}'s ROAS is {returns.loc[bottom, 'roas']:.2f}."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
search_days = channel_days[channel_days["channel_group"] == "Paid Search"].set_index("date")
days = search_days["spend"].resample("W").count()
search_weekly = search_days[["spend", "revenue", "organic"]].resample("W").sum()
search_weekly = search_weekly[days == 7]
search_model = smf.ols("revenue ~ spend + organic", data=search_weekly).fit()
search_model.params["spend"], search_model.conf_int().loc["spend"]
```

```python
rows = []
for group, group_days in channel_days.groupby("channel_group"):
    group_days = group_days.set_index("date")
    days = group_days["spend"].resample("W").count()
    weekly = group_days[["spend", "revenue", "organic"]].resample("W").sum()
    weekly = weekly[days == 7]
    model = smf.ols("revenue ~ spend + organic", data=weekly).fit()
    low, high = model.conf_int().loc["spend"]
    rows.append({"channel_group": group, "roas": weekly["revenue"].sum() / weekly["spend"].sum(),
                 "marginal_return": model.params["spend"], "return_low": low, "return_high": high})
returns = pd.DataFrame(rows).set_index("channel_group").sort_values("marginal_return", ascending=False)
returns
```

</details>